# Nimbus v1 — aspirate / dispense

**v1 shape:** `Nimbus` (device) → `NimbusDriver` (`HamiltonTCPClient`: TCP, introspection, resolved addresses) → `PIP` frontend → `NimbusPIPBackend` (pipette commands). After `setup()`, use **`nimbus.pip`** for moves; **`nimbus.driver`** for transport and low-level access.

In [1]:
from __future__ import annotations

import logging
import sys

from pylabrobot.hamilton.liquid_handlers.nimbus import Nimbus, NimbusSetupParams
from pylabrobot.hamilton.liquid_handlers.nimbus.pip_backend import (
  NimbusPIPAspirateParams,
  NimbusPIPDispenseParams,
)
from pylabrobot.hamilton.liquid_handlers.star.liquid_classes import get_star_liquid_class
from pylabrobot.resources import Cor_Axy_96_wellplate_500uL_Ub
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.hamilton import HamiltonTip, NimbusDeck, hamilton_96_tiprack_300uL_filter
from pylabrobot.resources.liquid import Liquid

logging.getLogger("pylabrobot").setLevel(logging.DEBUG)


def nimbus_star_water_hlcs(pip, channels: list[int], *, blow_out: bool = False, liquid: Liquid = Liquid.WATER):
  """One STAR liquid class per channel from mounted HamiltonTips (Water, jet=False)."""
  hlcs = []
  for ch in channels:
    tip = pip.head[ch].get_tip()
    if not isinstance(tip, HamiltonTip):
      raise TypeError(f"Channel {ch} needs HamiltonTip, got {type(tip).__name__}")
    hlc = get_star_liquid_class(
      tip_volume=tip.maximal_volume,
      is_core=False,
      is_tip=True,
      has_filter=tip.has_filter,
      liquid=liquid,
      jet=False,
      blow_out=blow_out,
    )
    if hlc is None:
      raise RuntimeError("No STAR liquid class for this tip/liquid/jet/blow_out combo.")
    hlcs.append(hlc)
  return hlcs

## Connect

`setup()`: TCP client registration, resolve **`nimbus_core`** / **`pipette`** (and optional **`door_lock`**), channel count, `InitializeSmartRoll` when needed, then **`PIP`** ready on **`nimbus.pip`**.

In [2]:
HOST = "192.168.100.100"
PORT = 2000
# Create deck to attach to Nimbus
deck = NimbusDeck()
tips = hamilton_96_tiprack_300uL_filter(name="tips_01", with_tips=True)
plate = Cor_Axy_96_wellplate_500uL_Ub(name="plate_01", with_lid=False)
deck.assign_child_resource(tips, location=Coordinate(x=305.750, y=126.537, z=128.620))
deck.assign_child_resource(plate, location=Coordinate(x=438.070, y=124.837, z=101.490))

nimbus = Nimbus(deck=deck, host=HOST, port=PORT)
await nimbus.setup(NimbusSetupParams(deck=deck))

2026-04-17 22:46:55,687 - pylabrobot.hamilton.tcp.client - INFO - Initializing Hamilton connection...
2026-04-17 22:46:55,697 - pylabrobot.hamilton.tcp.client - INFO - Registering Hamilton client...
2026-04-17 22:46:55,706 - pylabrobot.hamilton.tcp.client - INFO - Discovering Hamilton root objects...
2026-04-17 22:46:55,716 - pylabrobot.hamilton.tcp.client - INFO - Discovering Hamilton global objects...
2026-04-17 22:46:55,732 - pylabrobot.hamilton.tcp.client - INFO - Hamilton TCP client setup complete. Client ID: 2, globals: 1
2026-04-17 22:46:56,645 - pylabrobot.hamilton.tcp.interface_bundle - INFO - Nimbus interfaces: door_lock, nimbus_core, pipette
2026-04-17 22:46:57,469 - pylabrobot.hamilton.liquid_handlers.nimbus.driver - INFO - Channel configuration: 4 channels
2026-04-17 22:46:58,358 - pylabrobot.hamilton.liquid_handlers.nimbus.door - INFO - Door locked successfully
2026-04-17 22:46:58,763 - pylabrobot.hamilton.liquid_handlers.nimbus.pip_backend - INFO - Channel configuration 

## Interfaces


In [3]:
from pylabrobot.hamilton.liquid_handlers.nimbus.driver import nimbus_interface_specs_for_root

d = nimbus.driver
root = await d.introspection.get_object(d.get_root_object_addresses()[0])
for role, spec in nimbus_interface_specs_for_root(root.name).items():
  addr = getattr(d.nimbus_interfaces, role)
  if addr is None:
    print(f"{role}: {spec.path!r} → None")
  else:
    print(f"{role}: {spec.path!r} → {await d.resolve_path(spec.path)}")

nimbus_core: 'NimbusCORE' → 1:1:48896
pipette: 'NimbusCORE.Pipette' → 1:1:257
door_lock: 'NimbusCORE.DoorLock' → 1:1:268


## Pick up tips

Tip column slice + **`[:nc]`** (works for 4- or 8-channel heads). Adjust column to match tips on your rack.

In [4]:
# Example of a Channelized error from failed tip pickup
nc = nimbus.driver.pip.num_channels
print(f"channels: {nc}")
tip_pick = tips["A9:H9"][:nc]
await nimbus.pip.pick_up_tips(tip_pick)

channels: 4


2026-04-17 22:47:51,227 - pylabrobot.hamilton.tcp.client - ERROR - Hamilton COMMAND_EXCEPTION (action=0x5) on 4 channel(s): ch0: Tip Detected Not Correct Tip. (HcResult=0x0F4E) at (1, 1, 275) iface=1 action=6 (path=NimbusCORE.Channel, addr=1:1:275, method=[1:6] [1:6] PickupTip(zStartPosition: i32, zStopPosition: i32, zFinal: i32, volume: u32, colletCheck: i16) -> void), ch1: Tip Detected Not Correct Tip. (HcResult=0x0F4E) at (1, 1, 274) iface=1 action=6 (path=NimbusCORE.Channel, addr=1:1:274, method=[1:6] [1:6] PickupTip(zStartPosition: i32, zStopPosition: i32, zFinal: i32, volume: u32, colletCheck: i16) -> void), ch2: Tip Detected Not Correct Tip. (HcResult=0x0F4E) at (1, 1, 272) iface=1 action=6 (path=NimbusCORE.Channel, addr=1:1:272, method=[1:6] [1:6] PickupTip(zStartPosition: i32, zStopPosition: i32, zFinal: i32, volume: u32, colletCheck: i16) -> void), ch3: No Tip Picked Up. (HcResult=0x0F4B) at (1, 1, 273) iface=1 action=6 (path=NimbusCORE.Channel, addr=1:1:273, method=[1:6] [1:

ChannelizedError: ChannelizedError(errors={0: RuntimeError('Tip Detected Not Correct Tip. (HcResult=0x0F4E) at (1, 1, 275) iface=1 action=6'), 1: RuntimeError('Tip Detected Not Correct Tip. (HcResult=0x0F4E) at (1, 1, 274) iface=1 action=6'), 2: RuntimeError('Tip Detected Not Correct Tip. (HcResult=0x0F4E) at (1, 1, 272) iface=1 action=6'), 3: RuntimeError('No Tip Picked Up. (HcResult=0x0F4B) at (1, 1, 273) iface=1 action=6')}, raw_response=b'!\x00\x02\x00\x15\x00\x0f\x00\xa0\x000x0001.0x0001.0x0113:0x01,0x0006,0x0F4E;0x0001.0x0001.0x0112:0x01,0x0006,0x0F4E;0x0001.0x0001.0x0110:0x01,0x0006,0x0F4E;0x0001.0x0001.0x0111:0x01,0x0006,0x0F4B\x00', hoi_entries=[HcResultEntry(module_id=1, node_id=1, object_id=275, interface_id=1, action_id=6, result=3918), HcResultEntry(module_id=1, node_id=1, object_id=274, interface_id=1, action_id=6, result=3918), HcResultEntry(module_id=1, node_id=1, object_id=272, interface_id=1, action_id=6, result=3918), HcResultEntry(module_id=1, node_id=1, object_id=2

In [5]:
# Successful tip pickup
tip_pick = tips["E9:H9"][:nc]
await nimbus.pip.pick_up_tips(tip_pick)

2026-04-17 22:48:19,006 - pylabrobot.hamilton.liquid_handlers.nimbus.pip_backend - INFO - Picked up tips on channels [0, 1, 2, 3]


## Aspirate

`vols`, `liquid_height` (mm from well bottom), `flow_rates` — length **`nc`** (see `PIP` docs).

**Hamilton liquid classes:** `nimbus_star_water_hlcs` (defined with imports) builds per-channel classes; set **`blow_out_liquid_class`** to match partial vs blow-out STAR rows. **`swap_speed_mm_s`** sets leave-liquid speed (**mm/s**). Shared **`hlc_kw`** is reused for dispense below.

**swap_speed vs logs:** The `"Aspirated on channels …"` line does **not** print `swap_speed`. Values are still set on the `Aspirate` command: firmware uses **0.01 mm/s** per element, so **`15.0` mm/s → `1500`** on each active channel index. Use a TCP/trace capture or a temporary `send_command` wrapper if you need to see the raw command object.

In [6]:
ch = list(range(nc))
blow_out_liquid_class = False  # True → STAR blow-out / empty-tip style row
swap_speed_mm_s = 30.0

hlc_kw = dict(
  hamilton_liquid_classes=nimbus_star_water_hlcs(nimbus.pip, ch, blow_out=blow_out_liquid_class),
  swap_speed=[swap_speed_mm_s] * nc,
)

await nimbus.pip.aspirate(
  plate["A1:H1"][:nc],
  vols=[50.0] * nc,
  liquid_height=[2.0] * nc,
  backend_params=NimbusPIPAspirateParams(**hlc_kw),
)

2026-04-17 22:48:28,053 - pylabrobot.hamilton.liquid_handlers.nimbus.pip_backend - INFO - Aspirated on channels [0, 1, 2, 3]


## Dispense

Example: second column (`A2:H2` vs `A1:H1`). Same **`hlc_kw`** as aspirate so liquid class + swap speed stay aligned.

In [7]:
await nimbus.pip.dispense(
  plate["A2:H2"][:nc],
  vols=[50.0] * nc,
  liquid_height=[2.0] * nc,
  backend_params=NimbusPIPDispenseParams(**hlc_kw),
)

2026-04-17 22:48:31,971 - pylabrobot.hamilton.liquid_handlers.nimbus.pip_backend - INFO - Dispensed on channels [0, 1, 2, 3]


## Drop tips

To waste **`Trash`** sites (`waste_sites`), not back to the rack.

In [8]:
waste_sites = [deck.get_resource(f"default_long_{i}") for i in range(1, nc + 1)]

await nimbus.pip.drop_tips(tip_pick)

2026-04-17 22:48:38,246 - pylabrobot.hamilton.liquid_handlers.nimbus.pip_backend - INFO - Dropped tips on channels [0, 1, 2, 3]


## Cleanup

Park if you want the head stowed; unlock door when **`door_lock`** exists; **`stop()`** closes TCP.

In [9]:
await nimbus.park()
if nimbus.driver.door is not None:
    await nimbus.unlock_door()

await nimbus.stop()

2026-04-17 22:48:41,688 - pylabrobot.hamilton.liquid_handlers.nimbus.driver - INFO - Instrument parked successfully
2026-04-17 22:48:41,782 - pylabrobot.hamilton.liquid_handlers.nimbus.door - INFO - Door unlocked successfully
2026-04-17 22:48:41,783 - pylabrobot.io.socket - INFO - Closing connection to socket 192.168.100.100:2000
2026-04-17 22:48:41,784 - pylabrobot.hamilton.tcp.client - INFO - Hamilton TCP client stopped
